# Notebook 5: LLM Personalized Air Quality Advisory & Function Calling
**Author:** Guillén Concepción (Senior Data Scientist & MLOps Engineer)

This notebook demonstrates how to augment predicted PM2.5 forecasts from **Hopsworks Feature Store** and **XGBoost Regressor** with Large Language Models (LLMs) to generate personalized health advisories, outdoor exercise windows, and ventilation schedules for sensitive groups.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import DATA_DIR, DEFAULT_LOCATION
from src.llm_chain import AirQualityLLMChain

## 1. Load Latest Predictions Payload from Inference Pipeline

In [ ]:
pred_path = DATA_DIR / "latest_predictions.json"
if pred_path.exists():
    with open(pred_path, "r") as f:
        payload = json.load(f)
else:
    payload = {
        "location": DEFAULT_LOCATION,
        "current_us_aqi": 42,
        "current_pm2_5": 10.1,
        "forecast": [{"predicted_pm2_5": 9.5}, {"predicted_pm2_5": 11.2}, {"predicted_pm2_5": 8.4}]
    }

print(f"Location: {payload['location']}")
print(f"Current AQI: {payload['current_us_aqi']} | Current PM2.5: {payload['current_pm2_5']} ug/m3")

## 2. Define User Health Profiles & Run LLM Chain

In [ ]:
user_profiles = [
    {"name": "Active Outdoor Marathon Runner", "sensitive_group": "Athletes / Outdoor Training"},
    {"name": "Child with Asthma", "sensitive_group": "Pediatric Asthma / Respiratory Sensitive"},
    {"name": "Senior Citizen", "sensitive_group": "Elderly / Cardiovascular Health"}
]

chain = AirQualityLLMChain()

for profile in user_profiles:
    print(f"\n=========================================")
    print(f"👤 Advisory Profile: {profile['name']}")
    print(f"=========================================")
    advisory = chain.generate_personalized_recommendation(payload, user_profile=profile)
    
    if "llm_advisory" in advisory:
        print(advisory["llm_advisory"])
    else:
        print(f"📋 Summary             : {advisory['summary']}")
        print(f"🏃 Outdoor Activity   : {advisory['activity_guidance']}")
        print(f"🪟 Ventilation Advice : {advisory['ventilation_guidance']}")
        print(f"❤️ Sensitive Guidance  : {advisory['sensitive_group_guidance']}")